# Chapter 4: NLP Engineering Techniques with Scikit-learn

This notebook covers key NLP engineering techniques including text visualization, vectorization,
TF-IDF, model training and evaluation, handling imbalanced data, scikit-learn pipelines,
and SageMaker pipelines.

---
## Part 1 – Visualizing Data (Airbnb Reviews)

We start by downloading and exploring the Airbnb reviews dataset, then create a word cloud
to visualize the most common words in the reviews.

### Download the Airbnb Reviews Dataset

In [5]:
# !curl http://data.insideairbnb.com/united-states/ny/new-york-city/2022-12-04/data/reviews.csv.gz > airbnb-reviews.csv.gz
!curl https://data.insideairbnb.com/united-states/ny/new-york-city/2026-08-10/data/reviews.csv.gz > airbnb-reviews.csv.gz
!gzip -d airbnb-reviews.csv.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  114M  100  114M    0     0  24.3M      0  0:00:04  0:00:04 --:--:-- 24.3M
gzip: airbnb-reviews.csv already exists; do you wish to overwrite (y or n)? ^C


### Read and Explore the Data

In [ ]:
import pandas as pd

reviews = pd.read_csv("airbnb-reviews.csv")
print(reviews.shape)

### Sample 1000 Reviews

In [ ]:
sampled_reviews = reviews.sample(1000, random_state=42)
print(sampled_reviews.columns)

### View First Rows

In [ ]:
for index, row in sampled_reviews[:3].iterrows():
    print(f"Listing ID: {row.listing_id}")
    print(f"Comments: {row.comments}")
    print("---")

### Import Word Cloud Libraries

In [ ]:
!pip install wordcloud

In [ ]:
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS

### Combine All Review Text

In [ ]:
input_text = ""
for index, row in sampled_reviews.iterrows():
    input_text += str(row.comments)

### Generate Word Cloud

In [ ]:
wordcloud = WordCloud(
    width=3000,
    height=2000,
    random_state=42,
    background_color='white',
    colormap='winter',
    collocations=False,
    stopwords=STOPWORDS
).generate(input_text)

### Plot Word Cloud

In [ ]:
plt.figure(figsize=(40, 30))
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

---
## Part 2 – Vectorizing Text / CountVectorizer

We use scikit-learn's CountVectorizer to convert text into numerical features using bigrams,
then visualize the most common word pairs.

### Download Amazon Reviews Dataset

In [ ]:
import pandas as pd

!curl https://raw.githubusercontent.com/aws-samples/accelerated-data-lake/refs/heads/master/DataSources/AmazonReviews/amazon_reviews_us_sample.tsv > sample-amazon-reviews.tsv

df = pd.read_csv("sample-amazon-reviews.tsv", sep='\t', header=None)
df.head()

### CountVectorizer with Bigrams

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

count_vectorizer = CountVectorizer(stop_words='english', ngram_range=(2, 2))
sparse_matrix = count_vectorizer.fit_transform(sampled_reviews.comments)

### Build Word Frequency List

In [ ]:
sum_words = sparse_matrix.sum(axis=0)
words_freq = [(word, sum_words[0, idx]) for word, idx in count_vectorizer.vocabulary_.items()]
words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)

### Create Word Frequency DataFrame

In [ ]:
words_freq_df = pd.DataFrame(words_freq, columns=['bigram', 'count'])
words_freq_df.head()

### Drop First Row (Line Break Artifact)

In [ ]:
words_freq_df.drop(index=words_freq_df.index[0], axis=0, inplace=True)

### Plot Top 20 Bigrams

In [ ]:
words_freq_df[:20].plot.bar(
    x='bigram',
    y='count',
    title='Top 20 Bigrams in Airbnb Reviews',
    figsize=(12, 6)
)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## Part 3 – TF-IDF

TF-IDF (Term Frequency-Inverse Document Frequency) weights terms by how important they are
to a document relative to the entire corpus.

### Apply TF-IDF Vectorizer

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfIdf_vectorizer = TfidfVectorizer(use_idf=True)
tfIdf = tfIdf_vectorizer.fit_transform(sampled_reviews.comments)

### Build TF-IDF DataFrame and View Top Terms

In [ ]:
df_tfidf = pd.DataFrame(
    tfIdf.toarray(),
    columns=tfIdf_vectorizer.get_feature_names_out()
)

# Sum TF-IDF scores across all documents and sort
tfidf_scores = df_tfidf.sum(axis=0).sort_values(ascending=False)
print("Top 20 TF-IDF terms:")
print(tfidf_scores.head(20))

---
## Part 4 – Training and Evaluating a Model

We use Amazon Comprehend to detect language and sentiment, then train a Naive Bayes
classifier using the TF-IDF features.

### Initialize Amazon Comprehend Client

In [ ]:
import boto3

comprehend = boto3.client("comprehend")

### Language Detection Function

In [ ]:
def get_language_code(text):
    """Detect the dominant language of the given text."""
    try:
        response = comprehend.detect_dominant_language(Text=str(text)[:5000])
        languages = response['Languages']
        return languages[0]['LanguageCode'] if languages else 'unknown'
    except Exception as e:
        return 'unknown'

# Apply language detection to sampled reviews
sampled_reviews['Language'] = sampled_reviews.comments.apply(get_language_code)
sampled_reviews.head()

### Sentiment Detection Function

In [ ]:
def get_sentiment(text):
    """Detect the sentiment of the given text using Amazon Comprehend."""
    try:
        response = comprehend.detect_sentiment(
            Text=str(text)[:5000],
            LanguageCode='en'
        )
        return response['Sentiment']
    except Exception as e:
        return 'UNKNOWN'

# Apply sentiment detection to sampled reviews
sampled_reviews['Sentiment'] = sampled_reviews.comments.apply(get_sentiment)
sampled_reviews.head()

### Plot Sentiment Distribution

In [ ]:
sampled_reviews['Sentiment'].value_counts().plot.bar(
    title='Sentiment Distribution',
    figsize=(10, 6)
)
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

### Train Naive Bayes Classifier

In [ ]:
from sklearn.naive_bayes import MultinomialNB

classifier = MultinomialNB().fit(tfIdf[:800], sampled_reviews.Sentiment[:800])

### Predictions and Accuracy Score

In [ ]:
from sklearn.metrics import accuracy_score

predictions = classifier.predict(tfIdf[800:])
accuracy = accuracy_score(sampled_reviews.Sentiment[800:], predictions)
print(f"Accuracy: {accuracy:.4f}")

### Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(sampled_reviews.Sentiment[800:], predictions)
print("Confusion Matrix:")
print(cm)

---
## Part 5 – Handling Imbalanced Classification

Sentiment data is often heavily skewed toward positive. We address this by building
a balanced dataset with more negative and mixed samples.

### Sample Additional Reviews and Detect Sentiment

In [ ]:
# Sample 5000 additional reviews
additional_reviews = reviews.sample(5000, random_state=99)

# Detect language and sentiment
additional_reviews['Language'] = additional_reviews.comments.apply(get_language_code)
additional_reviews['Sentiment'] = additional_reviews.comments.apply(get_sentiment)

print(additional_reviews['Sentiment'].value_counts())

### Build Balanced DataFrame

In [ ]:
# Separate by sentiment
negative_reviews = additional_reviews[additional_reviews.Sentiment == 'NEGATIVE']
mixed_reviews = additional_reviews[additional_reviews.Sentiment == 'MIXED']
positive_reviews = additional_reviews[additional_reviews.Sentiment == 'POSITIVE'].sample(
    n=len(negative_reviews), random_state=42
)

# Combine into balanced dataset
final_df = pd.concat([mixed_reviews, negative_reviews, positive_reviews])
print(f"Balanced dataset shape: {final_df.shape}")
print(final_df['Sentiment'].value_counts())

### Save Balanced Dataset and Plot Distribution

In [ ]:
# Save to CSV
final_df.to_csv('balanced_reviews.csv', index=False)

# Plot distribution
final_df['Sentiment'].value_counts().plot.bar(
    title='Balanced Sentiment Distribution',
    figsize=(10, 6)
)
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

---
## Part 6 – Scikit-learn Pipeline

We build a complete NLP pipeline that chains together vectorization, TF-IDF transformation,
and classification into a single object.

### Encode Target Variable

In [ ]:
final_df['target'] = final_df.Sentiment.astype('category')
final_df.target = final_df.target.cat.codes
final_df[['Sentiment', 'target']].head()

### Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    final_df.comments,
    final_df.target,
    test_size=0.2,
    random_state=42
)

### Build the Pipeline

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline

pipeline_object = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('classifier', MultinomialNB())
])

### Fit the Pipeline

In [ ]:
pipeline_object.fit(X_train, y_train)

### Predict and Evaluate

In [ ]:
predictions = pipeline_object.predict(X_test)

# Confusion matrix
cm = confusion_matrix(y_test, predictions)
print("Confusion Matrix:")
print(cm)

# Accuracy
accuracy = accuracy_score(y_test, predictions)
print(f"\nAccuracy: {accuracy:.4f}")

### Sample Prediction with Sentiment Mapping

In [ ]:
sentiment_mapping_dict = {
    0: 'MIXED',
    1: 'NEGATIVE',
    2: 'NEUTRAL',
    3: 'POSITIVE'
}

# Test with a sample review
sample_text = ["The apartment was dirty and the host was rude"]
prediction = pipeline_object.predict(sample_text)
print(f"Predicted sentiment: {sentiment_mapping_dict[prediction[0]]}")

sample_text = ["Amazing place, great location, very clean!"]
prediction = pipeline_object.predict(sample_text)
print(f"Predicted sentiment: {sentiment_mapping_dict[prediction[0]]}")

---
## Part 7 – SageMaker Pipeline

We deploy our NLP model using a SageMaker Pipeline, which orchestrates the processing,
training, evaluation, and model registration steps.

### Initialize SageMaker Session

In [1]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

# Create a SageMaker session
sagemaker_session = Session()

# Get the execution role
role = get_execution_role()

region = sagemaker_session.boto_region_name
default_bucket = sagemaker_session.default_bucket()

print(f"Region: {region}")
print(f"Bucket: {default_bucket}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region: us-east-1
Bucket: sagemaker-us-east-1-862070281041


### Upload Balanced Reviews to S3

In [2]:
input_data_uri = sagemaker_session.upload_data(
    path='balanced_reviews.csv',
    bucket=default_bucket,
    key_prefix='nlp-pipeline/data'
)
print(f"Data uploaded to: {input_data_uri}")

Data uploaded to: s3://sagemaker-us-east-1-862070281041/nlp-pipeline/data/balanced_reviews.csv


### SageMaker Pipeline Steps

The full SageMaker pipeline includes:

1. **Processing Step** – Preprocesses the data (tokenization, vectorization, train/test split)
2. **Training Step** – Trains the scikit-learn model using the processed data
3. **Evaluation Step** – Evaluates model performance on the test set
4. **Condition Step** – Checks if accuracy meets the threshold before proceeding
5. **Register Model Step** – Registers the model in the SageMaker Model Registry if the condition passes

Each step is defined as a SageMaker Pipeline step object and chained together.

> **Note:** The three scripts (`preprocessing.py`, `train.py`, `evaluate.py`) referenced below
> must be placed in a `pipeline_scripts/` directory relative to this notebook.


#### Pipeline Parameters

Declare typed `ParameterInteger`, `ParameterString`, and `ParameterFloat` objects
so the pipeline can be re-run with different values without code changes.


In [3]:
from sagemaker.core.workflow.parameters import (
    ParameterInteger,
    ParameterString,
    ParameterFloat,
)

# ── Compute parameters ──────────────────────────────────────────────────
processing_instance_count = ParameterInteger(
    name="ProcessingInstanceCount",
    default_value=1,
)
processing_instance_type = ParameterString(
    name="ProcessingInstanceType",
    default_value="ml.m5.xlarge",
)
training_instance_count = ParameterInteger(
    name="TrainingInstanceCount",
    default_value=1,
)
training_instance_type = ParameterString(
    name="TrainingInstanceType",
    default_value="ml.m5.xlarge",
)

# ── Data / model parameters ───────────────────────────────────────────────
input_data = ParameterString(
    name="InputDataUrl",
    default_value=input_data_uri,   # uploaded in the cell above
)
model_approval_status = ParameterString(
    name="ModelApprovalStatus",
    default_value="PendingManualApproval",
)
accuracy_threshold = ParameterFloat(
    name="AccuracyThreshold",
    default_value=0.70,
)

print("Pipeline parameters defined.")


Pipeline parameters defined.


#### Step 1 – Processing Step

Runs `preprocessing.py` via a `ScriptProcessor`. Tokenises text, applies
TF-IDF, and splits data into train/test sets, writing the arrays to S3.


In [10]:
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core.image_uris import retrieve as retrieve_image_uri
from sagemaker.core.processing import ScriptProcessor
from sagemaker.core.shapes import (
    ProcessingInput, 
    ProcessingS3Input, 
    ProcessingOutput, 
    ProcessingS3Output
)
from sagemaker.mlops.workflow.steps import ProcessingStep

# V3 updated PipelineSession handles lazy compilation for pipeline steps
pipeline_session = PipelineSession()

# Resolve the managed scikit-learn container image
# Using .default_value prevents compile-time parsing from failing on parameters
sklearn_image_uri = retrieve_image_uri(
    framework="sklearn",
    region=region,
    version="1.2-1",
    image_scope="training",
    instance_type=processing_instance_type.default_value,
)

# Build processor mapped to the core pipeline context session
script_processor = ScriptProcessor(
    image_uri=sklearn_image_uri,
    command=["python3"],
    role=role,
    instance_type=processing_instance_type,
    instance_count=processing_instance_count,
    sagemaker_session=pipeline_session, 
    base_job_name="nlp-preprocessing",
)

# Intercept and return StepArguments lazily
preprocessing_step_args = script_processor.run(
    code="pipeline_scripts/preprocessing.py",
    inputs=[
        ProcessingInput(
            input_name="input-data",
            s3_input=ProcessingS3Input(
                s3_uri=input_data,
                local_path="/opt/ml/processing/input",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            ),
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            s3_output=ProcessingS3Output(
                s3_uri=f"s3://{default_bucket}/nlp-pipeline/processing/train",
                local_path="/opt/ml/processing/train",
                s3_upload_mode="EndOfJob",
            ),
        ),
        ProcessingOutput(
            output_name="test",
            s3_output=ProcessingS3Output(
                s3_uri=f"s3://{default_bucket}/nlp-pipeline/processing/test",
                local_path="/opt/ml/processing/test",
                s3_upload_mode="EndOfJob",
            ),
        ),
    ],
)

# Encapsulate step arguments into the MLOps pipeline structure
step_process = ProcessingStep(
    name="NLPPreprocessing",
    step_args=preprocessing_step_args,
)


[08/27/26 16:34:41] INFO     Defaulting to only available Python version: py3                     ]8;id=4260795;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=4260796;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#615\615]8;;\

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


#### Step 2 – Training Step

Uses `ModelTrainer` (SDK v3) with the managed scikit-learn container to train
a `MultinomialNB` classifier on the TF-IDF arrays produced by the Processing step.


In [11]:
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import SourceCode, Compute, InputData
from sagemaker.mlops.workflow.steps import TrainingStep

# Core configuration uses consolidated modules from sagemaker.core.training.configs
model_trainer = ModelTrainer(
    training_image=sklearn_image_uri,
    role=role,
    sagemaker_session=pipeline_session, # Binds to our active pipeline recording context
    source_code=SourceCode(
        source_dir="pipeline_scripts",
        entry_script="train.py",
    ),
    compute=Compute(
        instance_type=training_instance_type,
        instance_count=training_instance_count,
    ),
    base_job_name="nlp-training",
)

# Wire the train-channel downstream directly from the Processing step properties payload
training_step_args = model_trainer.train(
    input_data_config=[
        InputData(
            channel_name="train",
            data_source=step_process.properties.ProcessingOutputConfig
                .Outputs["train"].S3Output.S3Uri,
        )
    ]
)

# Build the final MLOps pipeline node mapping dependencies cleanly
step_train = TrainingStep(
    name="NLPTraining",
    step_args=training_step_args,
    depends_on=[step_process],
)

print("Training step defined:", step_train.name)


[08/27/26 16:35:58] INFO     StoppingCondition not provided. Using default:                         ]8;id=4260803;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4260804;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     OutputDataConfig not provided. Using default:                          ]8;id=4260810;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4260811;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#153\153]8;;\
                             s3_output_path='s3://sagemaker-us-east-1-862070281041/nlp-training'                   
                             kms_key_id=None compression_type='GZIP'                                               

                    INFO     Training image URI:                                               ]8;id=4260818;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=4260819;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-lea                     
                             rn:1.2-1-cpu-py3                                                                      

Training step defined: NLPTraining


#### Step 3 – Evaluation Step

Runs `evaluate.py` via a second `ScriptProcessor`. Loads the trained model
artifact and the test arrays, computes accuracy, and writes `evaluation.json`
to S3 where the Condition Step can read it.


In [13]:
from sagemaker.core.processing import ScriptProcessor
from sagemaker.core.shapes import (
    ProcessingInput, 
    ProcessingS3Input, 
    ProcessingOutput, 
    ProcessingS3Output
)
from sagemaker.core.workflow.properties import PropertyFile
from sagemaker.mlops.workflow.steps import ProcessingStep

# Processor instantiation utilizing our unified v3 script wrapper
eval_processor = ScriptProcessor(
    image_uri=sklearn_image_uri,
    command=["python3"],
    role=role,
    instance_type=processing_instance_type,
    instance_count=processing_instance_count,
    sagemaker_session=pipeline_session,
    base_job_name="nlp-evaluation",
)

# PropertyFile lets the Condition Step read accuracy metrics from evaluation.json
evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

# Run method captures our step configurations lazily into an internal graph layout
eval_step_args = eval_processor.run(
    code="pipeline_scripts/evaluate.py",
    inputs=[
        ProcessingInput(
            input_name="model",
            s3_input=ProcessingS3Input(
                # Fixed to map directly to the AWS dynamic API schema definition
                s3_uri=step_train.properties.ModelArtifacts.S3ModelArtifacts,
                local_path="/opt/ml/processing/model",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            ),
        ),
        ProcessingInput(
            input_name="test-data",
            s3_input=ProcessingS3Input(
                s3_uri=step_process.properties.ProcessingOutputConfig
                    .Outputs["test"].S3Output.S3Uri,
                local_path="/opt/ml/processing/test",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            ),
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            s3_output=ProcessingS3Output(
                s3_uri=f"s3://{default_bucket}/nlp-pipeline/evaluation",
                local_path="/opt/ml/processing/evaluation",
                s3_upload_mode="EndOfJob",
            ),
        )
    ],
)

# Bind the step arguments and map the properties file configuration into MLOps
step_eval = ProcessingStep(
    name="NLPEvaluation",
    step_args=eval_step_args,
    property_files=[evaluation_report],
    depends_on=[step_train],
)

print("Evaluation step defined:", step_eval.name)


Evaluation step defined: NLPEvaluation


#### Step 4 – Condition Step

Reads the `accuracy` value from `evaluation.json` and only proceeds to
model registration if accuracy ≥ `AccuracyThreshold` (default 0.70).


In [15]:
from sagemaker.core.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.core.workflow.functions import JsonGet
from sagemaker.mlops.workflow.condition_step import ConditionStep

# Extract the scalar accuracy value from the evaluation report via the core functional engine
accuracy_value = JsonGet(
    step_name=step_eval.name,
    property_file=evaluation_report,
    json_path="binary_classification_metrics.accuracy.value",
)

# Core condition checks use native type definitions mapping parameters safely
accuracy_condition = ConditionGreaterThanOrEqualTo(
    left=accuracy_value,
    right=accuracy_threshold,
)

# ── Register Model Step (defined next) will be wired in here ──────────────
# step_cond is completed after step_register is defined below.
print("Accuracy condition defined.")


Accuracy condition defined.


#### Step 5 – Register Model Step

If the Condition Step passes, registers the trained model in the
SageMaker Model Registry under the `NLPSentimentClassifier` model package group.


In [28]:
from sagemaker.core.resources import ModelPackage
from sagemaker.core.shapes import InferenceSpecification, ContainersItem
from sagemaker.core.workflow.functions import Join
from sagemaker.mlops.workflow.model_step import ModelStep
from sagemaker.mlops.workflow.condition_step import ConditionStep

# 1. Map configurations using formal v3 shape wrappers and the workflow Join helper
inference_spec = InferenceSpecification(
    containers=[
        ContainersItem(
            image=sklearn_image_uri,
            # Wrap the properties variable inside Join to bypass local Pydantic string validation
            model_data_url=Join(on="", values=[step_train.properties.ModelArtifacts.S3ModelArtifacts]),
        )
    ],
    supported_content_types=["text/csv"],
    supported_response_mime_types=["application/json"],
    supported_realtime_inference_instance_types=["ml.m5.large", "ml.m5.xlarge"],
    supported_transform_instance_types=["ml.m5.large"],
)

# 2. Build the formal SageMaker Registry resource schema object
nlp_model_package = ModelPackage(
    model_package_group_name="NLPSentimentClassifier",
    model_package_description="NLP Sentiment Classifier Model migrated to SageMaker v3",
    inference_specification=inference_spec,
)

# 3. Extract step args natively via the core context registration handler
register_args = nlp_model_package.register(
    approval_status=model_approval_status,
    role=role,
    sagemaker_session=pipeline_session, # Binds the active pipeline context recording framework
)

# 4. Encapsulate the structured transaction cleanly inside the workflow model step
step_register = ModelStep(
    name="NLPRegisterModel",
    step_args=register_args,
)

# 5. Wire the finished registration configuration into the branch layout
step_cond = ConditionStep(
    name="NLPAccuracyCondition",
    conditions=[accuracy_condition],
    if_steps=[step_register],   # Executes model registration if condition evaluates to true
    else_steps=[],              # Graceful no-op block if threshold validation fails
)

print("Register Model step defined:", step_register.name)
print("Condition step defined:",       step_cond.name)


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:10                                                                                   │
│                                                                                                  │
│    7 # 1. Map configurations using formal v3 shape wrappers and the workflow Join helper         │
│    8 inference_spec = InferenceSpecification(                                                    │
│    9 │   containers=[                                                                            │
│ ❱ 10 │   │   ContainersItem(                                                                     │
│   11 │   │   │   image=sklearn_image_uri,                                                        │
│   12 │   │   │   # Wrap the properties variable inside Join to bypass local Pydantic string v    │
│   13 │   │   │   model_data_url=Join(on="", values=[step_train.properties.ModelArtifacts.S3Mo    │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/pydantic/main.py:263 in __init__                         │
│                                                                                                  │
│    260 │   │   """                                                                               │
│    261 │   │   # `__tracebackhide__` tells pytest and some other tools to omit this function fr  │
│    262 │   │   __tracebackhide__ = True                                                          │
│ ❱  263 │   │   validated_self = self.__pydantic_validator__.validate_python(data, self_instance  │
│    264 │   │   if self is not validated_self:                                                    │
│    265 │   │   │   warnings.warn(                                                                │
│    266 │   │   │   │   'A custom validator is returning a value other than `self`.\n'            │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ValidationError: 1 validation error for ContainersItem
model_data_url
  Input should be a valid string [type=string_type, input_value=Join(on='', values=[{'_st...], '__str__': 
'S3Uri'}]), input_type=Join]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

## try x

In [ ]:
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core.image_uris import retrieve as retrieve_image_uri
from sagemaker.core.processing import ScriptProcessor
from sagemaker.core.shapes import (
    ProcessingInput, 
    ProcessingS3Input, 
    ProcessingOutput, 
    ProcessingS3Output
)
from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import SourceCode, Compute, InputData
from sagemaker.core.workflow.properties import PropertyFile
from sagemaker.core.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.core.workflow.functions import JsonGet
from sagemaker.mlops.workflow.steps import ProcessingStep, TrainingStep
from sagemaker.mlops.workflow.model_step import ModelStep  

from sagemaker.mlops.workflow.condition_step import ConditionStep
from sagemaker.mlops.workflow.pipeline import Pipeline

# ── 0. GLOBAL pipeline CONTEXT ─────────────────────────────────────────────
pipeline_session = PipelineSession()

# Resolve the managed container image safely
sklearn_image_uri = retrieve_image_uri(
    framework="sklearn",
    region=region,
    version="1.2-1",
    image_scope="training",
    instance_type=processing_instance_type.default_value,
)

In [30]:
# ── 1. PROCESSING STEP ─────────────────────────────────────────────────────
script_processor = ScriptProcessor(
    image_uri=sklearn_image_uri,
    command=["python3"],
    role=role,
    instance_type=processing_instance_type,
    instance_count=processing_instance_count,
    sagemaker_session=pipeline_session,
    base_job_name="nlp-preprocessing",
)

preprocessing_step_args = script_processor.run(
    code="pipeline_scripts/preprocessing.py",
    inputs=[
        ProcessingInput(
            input_name="input-data",
            s3_input=ProcessingS3Input(
                s3_uri=input_data,
                local_path="/opt/ml/processing/input",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            ),
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            s3_output=ProcessingS3Output(
                s3_uri=f"s3://{default_bucket}/nlp-pipeline/processing/train",
                local_path="/opt/ml/processing/train",
                s3_upload_mode="EndOfJob",
            ),
        ),
        ProcessingOutput(
            output_name="test",
            s3_output=ProcessingS3Output(
                s3_uri=f"s3://{default_bucket}/nlp-pipeline/processing/test",
                local_path="/opt/ml/processing/test",
                s3_upload_mode="EndOfJob",
            ),
        ),
    ],
)

step_process = ProcessingStep(
    name="NLPPreprocessing",
    step_args=preprocessing_step_args,
)

[08/27/26 16:44:52] INFO     Defaulting to only available Python version: py3                     ]8;id=4260839;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=4260840;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#615\615]8;;\

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [31]:
# ── 2. TRAINING STEP ───────────────────────────────────────────────────────
model_trainer = ModelTrainer(
    training_image=sklearn_image_uri,
    role=role,
    sagemaker_session=pipeline_session,
    source_code=SourceCode(
        source_dir="pipeline_scripts",
        entry_script="train.py",
    ),
    compute=Compute(
        instance_type=training_instance_type,
        instance_count=training_instance_count,
    ),
    base_job_name="nlp-training",
)

training_step_args = model_trainer.train(
    input_data_config=[
        InputData(
            channel_name="train",
            data_source=step_process.properties.ProcessingOutputConfig
                .Outputs["train"].S3Output.S3Uri,
        )
    ]
)

step_train = TrainingStep(
    name="NLPTraining",
    step_args=training_step_args,
    depends_on=[step_process],
)

[08/27/26 16:45:15] INFO     StoppingCondition not provided. Using default:                         ]8;id=4260845;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4260846;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#128\128]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     OutputDataConfig not provided. Using default:                          ]8;id=4260851;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=4260852;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#153\153]8;;\
                             s3_output_path='s3://sagemaker-us-east-1-862070281041/nlp-training'                   
                             kms_key_id=None compression_type='GZIP'                                               

                    INFO     Training image URI:                                               ]8;id=4260857;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=4260858;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-lea                     
                             rn:1.2-1-cpu-py3                                                                      

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [32]:
# ── 3. EVALUATION STEP ─────────────────────────────────────────────────────
eval_processor = ScriptProcessor(
    image_uri=sklearn_image_uri,
    command=["python3"],
    role=role,
    instance_type=processing_instance_type,
    instance_count=processing_instance_count,
    sagemaker_session=pipeline_session,
    base_job_name="nlp-evaluation",
)

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

eval_step_args = eval_processor.run(
    code="pipeline_scripts/evaluate.py",
    inputs=[
        ProcessingInput(
            input_name="model",
            s3_input=ProcessingS3Input(
                s3_uri=step_train.properties.ModelArtifacts.S3ModelArtifacts,
                local_path="/opt/ml/processing/model",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            ),
        ),
        ProcessingInput(
            input_name="test-data",
            s3_input=ProcessingS3Input(
                s3_uri=step_process.properties.ProcessingOutputConfig
                    .Outputs["test"].S3Output.S3Uri,
                local_path="/opt/ml/processing/test",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            ),
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            s3_output=ProcessingS3Output(
                s3_uri=f"s3://{default_bucket}/nlp-pipeline/evaluation",
                local_path="/opt/ml/processing/evaluation",
                s3_upload_mode="EndOfJob",
            ),
        )
    ],
)

step_eval = ProcessingStep(
    name="NLPEvaluation",
    step_args=eval_step_args,
    property_files=[evaluation_report],
    depends_on=[step_train],
)

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


In [34]:
from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.mlops.workflow.model_step import ModelStep
from sagemaker.mlops.workflow.condition_step import ConditionStep

# 1. Define ModelBuilder pointing directly to the dynamic training output token
model_builder = ModelBuilder(
    image_uri=sklearn_image_uri,
    s3_model_data_url=step_train.properties.ModelArtifacts.S3ModelArtifacts, # Handled automatically inside serve
    role_arn=role,
    sagemaker_session=pipeline_session, # Safely captures recording state variables
)

# 2. Extract step args natively using the model_builder utility layer
register_args = model_builder.register(
    model_package_group_name="NLPSentimentClassifier",
    content_types=["application/json"],
    response_types=["application/json"],
    inference_instances=["ml.m5.xlarge"],
    approval_status=model_approval_status,
)

# 3. Wrap register arguments into the MLOps pipeline step engine
step_register = ModelStep(
    name="NLPRegisterModel",
    step_args=register_args,
)

# 4. Wire the registration step dynamically into the true conditional branch
step_cond = ConditionStep(
    name="NLPAccuracyCondition",
    conditions=[accuracy_condition],
    if_steps=[step_register],   # Executes model registration if condition evaluates to true
    else_steps=[],              # Graceful no-op block if threshold validation fails
)

print("Register Model step defined:", step_register.name)
print("Condition step defined:",       step_cond.name)


[08/27/26 16:46:53] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=4260863;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=4260864;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#340\340]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=4260869;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=4260870;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#374\374]8;;\

Register Model step defined: NLPRegisterModel
Condition step defined: NLPAccuracyCondition


In [38]:
# ── 5. PIPELINE DEFINTION GRAPH ───────────────────────────────────────────
pipeline = Pipeline(
    name="nlp-sentiment-pipeline",
    parameters=[
        processing_instance_type,
        processing_instance_count,
        training_instance_type,
        training_instance_count,
        input_data,
        accuracy_threshold,
        model_approval_status,
    ],
    # steps=[step_process, step_train, step_eval, step_cond]
    steps=[step_process, step_train, step_register],
    sagemaker_session=pipeline_session,
)

# Commit the topology out to AWS
pipeline.upsert(role_arn=role)
print("Pipeline successfully updated and verified under SageMaker v3 SDK.")

[08/27/26 17:20:15] WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=4260908;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=4260909;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[08/27/26 17:20:16] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=4260914;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=4260915;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=4260920;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=4260921;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=4260926;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=4260927;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[08/27/26 17:20:17] WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=4260932;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=4260933;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=4260938;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=4260939;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

Pipeline successfully updated and verified under SageMaker v3 SDK.


In [39]:
# Trigger the execution run
# Note: You can optionally pass runtime parameter overrides inside a dictionary
execution = pipeline.start()
    # parameters={
    #     "accuracy_threshold": 0.85,
    #     "model_approval_status": "PendingManualApproval"
    # }
# )

print(f"Pipeline execution started successfully!")
print(f"Execution ARN: {execution.arn}")

Pipeline execution started successfully!
Execution ARN: arn:aws:sagemaker:us-east-1:862070281041:pipeline/nlp-sentiment-pipeline/execution/qzonz28ddk7p


In [ ]:

# Wait for the execution graph to finish processing (Optional blocking call)
print("Waiting for steps to complete processing across AWS infrastructure...")
execution.wait()

# Print out the final execution status breakdown
print(f"Execution completed with status: {execution.describe()['PipelineExecutionStatus']}")

---
## Cleanup & Notes

### EventBridge Integration

SageMaker Pipelines integrates with Amazon EventBridge to emit events at each step
transition. You can configure EventBridge rules to:

- Send notifications when a pipeline succeeds or fails
- Trigger downstream workflows (e.g., deploy model to endpoint)
- Log pipeline execution metadata

### Cleanup

To avoid ongoing charges, remember to:

1. Delete the SageMaker Pipeline when no longer needed
2. Remove any S3 data and model artifacts
3. Deregister models from the Model Registry
4. Delete any EventBridge rules created for pipeline monitoring
5. Stop or delete any SageMaker endpoints deployed from this pipeline

In [ ]:
# Cleanup: Delete the pipeline (uncomment to execute)
# import boto3
# sm_client = boto3.client('sagemaker')
# sm_client.delete_pipeline(PipelineName=pipeline_name)
# print(f"Pipeline '{pipeline_name}' deleted.")